# Build silver service
This contains service calls, appointments, hours, equipment, tasks, costs, and labor estimates from Wennsoft.

In [1]:
# Attach the schema-enabled Lakehouse containing the Bronze shortcuts and
# make it the default Lakehouse before running this notebook.
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
METADATA_SCHEMA = "metadata"
COMPANY_KEY = "NAC"  # Change when processing another GP company database.
COST_CODE_LABELS = {
    1: "Equipment",
    2: "Material",
    4: "Subcontractor",
    5: "Other",
    6: "Labor 1",
    7: "Labor 2",
    8: "Labor 3"
}

def q(identifier: str) -> str:
    return f"`{identifier.replace('`', '``')}`"

def sql_string(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"

def table_exists(schema_name: str, table_name: str) -> bool:
    try:
        return spark.catalog.tableExists(f"{schema_name}.{table_name}")
    except Exception:
        return False

def require_tables(table_names: list[str], schema_name: str = BRONZE_SCHEMA) -> None:
    missing = [name for name in table_names if not table_exists(schema_name, name)]
    if missing:
        raise RuntimeError(
            f"Missing tables in {schema_name}: {', '.join(missing)}. "
            "Confirm the OneLake shortcuts and exact table names."
        )

company = sql_string(COMPANY_KEY)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(SILVER_SCHEMA)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(GOLD_SCHEMA)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(METADATA_SCHEMA)}")
print(f"Company: {COMPANY_KEY}; Bronze schema: {BRONZE_SCHEMA}")

StatementMeta(, 85b89331-b16a-48c8-a299-071d8de3741e, 3, Finished, Available, Finished, False)

Company: NAC; Bronze schema: bronze


In [2]:
MIN_SERVICE_DATE = "2021-01-01"
# SV_Costs is a SQL Server view and cannot be mirrored. Its two
# physical transaction sources are materialized below instead.
require_tables(["SV000810", "SV000815", "SV00300", "SV00301", "SV00302",
                "SV00500", "SV00501", "SV00585"])

# dbo.wsGetWSCostCode is a SQL Server scalar function, so Spark cannot
# execute it. Code 6 is confirmed as LABOR by the source reporting view.
# Add the remaining values returned by dbo.wsGetWSCostCode in your GP
# database; every transaction is retained even when a label is absent.
COST_CODE_LABELS = {6: "LABOR"}
cost_label_case = "CASE " + " ".join(
    f"WHEN TRY_CAST(s.WS_Cost_Code AS INT) = {int(code)} THEN {sql_string(label)}"
    for code, label in COST_CODE_LABELS.items()
) + " ELSE CONCAT('COST CODE ', CAST(TRY_CAST(s.WS_Cost_Code AS INT) AS STRING)) END"

StatementMeta(, 85b89331-b16a-48c8-a299-071d8de3741e, 4, Finished, Available, Finished, False)

## Create SV_Calls
This recreates the SV_Calls view from WennSoft.

In [3]:
from pyspark.sql import functions as F

open_cost_df = (
    spark.table(f"{BRONZE_SCHEMA}.SV000810")
    .filter("COALESCE(REVSETRX, 0) = 0 AND UPPER(TRIM(Dynamics_Status)) = 'OPEN'")
    .withColumn("source_record_area", F.lit("OPEN"))
)
history_cost_df = (
    spark.table(f"{BRONZE_SCHEMA}.SV000815")
    .filter("COALESCE(REVSETRX, 0) = 0")
    .withColumn("source_record_area", F.lit("HISTORY"))
)
# Current and history tables can differ slightly between WennSoft
# releases. Union by name preserves the shared reporting columns and
# tolerates source-only additions.
cost_source_df = open_cost_df.unionByName(
    history_cost_df, allowMissingColumns=True
).dropDuplicates()
cost_source_df.createOrReplaceTempView("_service_cost_source")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.service_cost_transaction
USING DELTA
AS
SELECT
    {company} AS company_key,
    s.source_record_area,
    TRIM(s.Service_Call_ID) AS work_number,
    TRIM(s.Call_Invoice_Number) AS invoice_number,
    TRY_CAST(s.JRNENTRY AS BIGINT) AS journal_number,
    TRIM(s.Divisions) AS division_code,
    CAST(s.UNITCOST AS DECIMAL(19,5)) AS unit_cost,
    TRIM(s.Technician_ID) AS technician_code,
    TRIM(s.Technician_Team) AS technician_team,
    TRIM(s.Reference_TRX_Number) AS reference_transaction_number,
    TRIM(s.CUSTNMBR) AS customer_code,
    TRIM(s.ADRSCODE) AS location_code,
    TRY_CAST(s.SEQNUMBR AS BIGINT) AS sequence_number,
    TRIM(s.WS_Job_Name) AS job_name,
    TRIM(s.Job_Type) AS job_type,
    TRY_CAST(s.WS_Cost_Code AS INT) AS cost_code,
    {cost_label_case} AS cost_label,
    NULLIF(TRIM(s.Cost_Code_Description), '') AS cost_description,
    TRIM(s.Task_Code) AS task_code,
    TRY_CAST(s.DOCTYPE AS INT) AS document_code,
    CAST(s.DOCAMNT AS DECIMAL(19,5)) AS document_amount,
    CAST(s.DOCDATE AS DATE) AS document_date,
    NULLIF(TRIM(s.TRXDSCRN), '') AS transaction_description,
    TRY_CAST(s.Billable_Check_Box AS INT) AS billable_flag,
    TRY_CAST(s.TAXABLE AS INT) AS taxable_flag,
    TRIM(s.TRXSOURC) AS transaction_source,
    CAST(s.Billing_Amount AS DECIMAL(19,5)) AS billing_amount,
    TRIM(s.VENDORID) AS vendor_code,
    TRIM(s.VENDNAME) AS vendor_name,
    TRIM(s.Equipment_ID) AS equipment_code,
    TRIM(s.Equipment_Type) AS equipment_type,
    TRIM(s.Dynamics_Status) AS dynamics_status,
    TRIM(s.Wennsoft_Status) AS wennsoft_status,
    TRIM(s.EMPLOYID) AS employee_code_text,
    TRIM(s.EMPLNAME) AS employee_name,
    CAST(s.OH_Amount AS DECIMAL(19,5)) AS overhead_amount,
    CAST(s.EXTDCOST AS DECIMAL(19,5)) AS extended_cost,
    CAST(s.WS_Extended_Cost AS DECIMAL(19,5)) AS ws_extended_cost,
    CAST(s.TRXQTY AS DECIMAL(19,5)) AS transaction_quantity,
    TRIM(s.UOFM) AS unit_of_measure,
    TRIM(s.ITEMNMBR) AS item_number,
    TRIM(s.ITEMDESC) AS item_description,
    CAST(s.UNITPRCE AS DECIMAL(19,5)) AS unit_price,
    CAST(s.TRXHRUNT AS DOUBLE) / 100.0 AS actual_hours,
    CAST(s.TRXHRUNTBILL AS DOUBLE) / 100.0 AS billed_hours,
    CAST(s.WS_Committed_Cost AS DECIMAL(19,5)) AS committed_cost,
    CAST(sc.DATE1 AS DATE) AS service_date,
    NULLIF(TRIM(sc.Contract_Number), '') AS contract_number,
    TRY_CAST(sc.WSCONTSQ AS BIGINT) AS contract_sequence,
    CASE WHEN CAST(cm.Contract_Start_Date AS DATE) > DATE'1900-01-01'
         THEN CAST(cm.Contract_Start_Date AS DATE) END AS contract_start_date,
    CASE WHEN CAST(cm.Contract_Expiration_Date AS DATE) > DATE'1900-01-01'
         THEN CAST(cm.Contract_Expiration_Date AS DATE) END AS contract_end_date,
    TRY_CAST(s.DEX_ROW_ID AS BIGINT) AS source_row_id,
    CAST(s.WS_Transaction_Note AS STRING) AS transaction_note,
    CURRENT_TIMESTAMP() AS _processed_at
FROM _service_cost_source s
INNER JOIN {q(BRONZE_SCHEMA)}.SV00300 sc
  ON s.Service_Call_ID = sc.Service_Call_ID
LEFT JOIN {q(BRONZE_SCHEMA)}.SV00500 cm
  ON s.CUSTNMBR = cm.CUSTNMBR
 AND s.ADRSCODE = cm.ADRSCODE
 AND sc.Contract_Number = cm.Contract_Number
 AND sc.WSCONTSQ = cm.WSCONTSQ
""")

StatementMeta(, 85b89331-b16a-48c8-a299-071d8de3741e, 5, Finished, Available, Finished, False)

DataFrame[]

## Service calls and hours

In [4]:
spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.service_call
USING DELTA
AS
SELECT
    {company} AS company_key,
    TRIM(s.Service_Call_ID) AS work_number,
    TRIM(s.ADRSCODE) AS location_code,
    TRIM(s.CUSTNMBR) AS customer_code,
    CASE WHEN TRIM(s.Contract_Number) <> '' THEN TRUE ELSE FALSE END AS contract_flag,
    TRIM(s.Contract_Number) AS contract_code,
    TRY_CAST(s.WSCONTSQ AS BIGINT) AS contract_sequence,
    TRIM(s.Service_Description) AS work_description,
    TRIM(s.Technician_ID) AS technician_primary,
    TRIM(s.Technician) AS technician_assigned,
    TRIM(s.Resolution_ID) AS status_code,
    TRIM(s.Type_Call_Short) AS type_code,
    TRIM(s.Status_of_Call) AS status,
    CAST(s.Date_of_Service_Call AS DATE) AS call_date,
    CASE WHEN CAST(s.Completion_Date AS DATE) > DATE'1900-01-02'
         THEN CAST(s.Completion_Date AS DATE) END AS completion_date,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.SV00300 s
WHERE TRIM(s.Service_Call_ID) <> ''
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.service_call_hours
USING DELTA
AS
SELECT
    {company} AS company_key,
    TRIM(Service_Call_ID) AS work_number,
    SUM(CAST(Actual_Hours AS DOUBLE) / 100.0) AS actual_hours,
    SUM(CAST(Estimate_Hours AS DOUBLE) / 100.0) AS budget_hours,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.SV00301
WHERE TRIM(Service_Call_ID) <> ''
GROUP BY TRIM(Service_Call_ID)
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.service_call_task_budget
USING DELTA
AS
SELECT
    {company} AS company_key,
    TRIM(CUSTNMBR) AS customer_code,
    TRIM(ADRSCODE) AS location_code,
    TRIM(Service_Call_ID) AS work_number,
    TRIM(Task_Status) AS call_status,
    CASE WHEN CAST(Completion_Date AS DATE) > DATE'1900-01-02'
         THEN CAST(Completion_Date AS DATE) END AS completion_date,
    CAST(Estimate_Hours AS DOUBLE) / 100.0 AS budget_hours,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.SV00302
WHERE TRIM(Equipment_ID) = 'HOURS'
""")

StatementMeta(, 85b89331-b16a-48c8-a299-071d8de3741e, 6, Finished, Available, Finished, False)

DataFrame[]

## Appointments and actual labor

In [5]:
spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.service_appointment
USING DELTA
AS
SELECT
    {company} AS company_key,
    TRIM(a.Service_Call_ID) AS work_number,
    TRIM(a.Appointment) AS appointment_number,
    TRIM(a.Appointment_Description) AS appointment_description,
    TRIM(a.Appointment_Status) AS appointment_status,
    TRIM(a.Technician) AS technician_username,
    TRY_CAST(a.WS_Cost_Code AS INT) AS cost_code,
    TRIM(a.WS_Job_Number) AS job_number,
    CASE WHEN CAST(a.Task_Date AS DATE) > DATE'1900-01-01'
         THEN CAST(a.Task_Date AS DATE) END AS scheduled_date,
    CASE WHEN CAST(a.Completion_Date AS DATE) > DATE'1900-01-02'
         THEN CAST(a.Completion_Date AS DATE) END AS completion_date,
    CASE WHEN a.Estimate_Hours > 0
         THEN CAST(a.Estimate_Hours AS DOUBLE) / 100.0 END AS estimated_hours,
    CASE WHEN a.Actual_Hours > 0
         THEN CAST(a.Actual_Hours AS DOUBLE) / 100.0 END AS actual_hours,
    CASE WHEN CAST(a.CREATDDT AS DATE) > DATE'1900-01-01'
         THEN CAST(a.CREATDDT AS DATE) END AS created_date,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.SV00301 a
WHERE TRIM(a.Service_Call_ID) <> ''
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.service_labor_actual
USING DELTA
AS
SELECT
    {company} AS company_key,
    work_number,
    division_code,
    customer_code,
    location_code,
    actual_hours,
    employee_name,
    TRY_CAST(employee_code_text AS BIGINT) AS employee_code,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(SILVER_SCHEMA)}.service_cost_transaction
WHERE cost_code = 6
""")

StatementMeta(, 85b89331-b16a-48c8-a299-071d8de3741e, 7, Submitted, Running, Running, True)

DataFrame[]

## Equipment and maintenance tasks

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.service_call_equipment
USING DELTA
AS
SELECT DISTINCT
    {company} AS company_key,
    TRIM(Service_Call_ID) AS work_number,
    TRIM(Equipment_ID) AS equipment_code,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(BRONZE_SCHEMA)}.SV00302
WHERE TRIM(Service_Call_ID) <> ''
  AND TRIM(Equipment_ID) NOT IN ('', 'HOURS')
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.service_call_equipment_task
USING DELTA
AS
WITH ranked AS (
    SELECT
        {company} AS company_key,
        TRIM(Service_Call_ID) AS work_number,
        TRIM(Equipment_ID) AS equipment_code,
        TRIM(Task_code) AS task_code,
        ROW_NUMBER() OVER (
            PARTITION BY TRIM(Service_Call_ID), TRIM(Equipment_ID)
            ORDER BY
                CASE
                    WHEN SUBSTRING(TRIM(Task_code), 1, 3) IN ('AN-', 'SU-', 'WI-') THEN 1
                    WHEN SUBSTRING(TRIM(Task_code), 1, 3) IN ('IN-', 'RN-', 'RT-') THEN 2
                    WHEN SUBSTRING(TRIM(Task_code), 1, 3) = 'WAT' THEN 3
                    ELSE 4
                END,
                TRIM(Task_code)
        ) AS rn
    FROM {q(BRONZE_SCHEMA)}.SV00302
    WHERE TRIM(Task_code) NOT IN ('', 'HOURS', 'DEFAULT')
)
SELECT
    company_key,
    work_number,
    equipment_code,
    task_code,
    CASE
        WHEN SUBSTRING(task_code, 1, 3) IN ('AN-', 'SU-', 'WI-') THEN 'Annual maintenance'
        WHEN SUBSTRING(task_code, 1, 3) IN ('RT-', 'IN-', 'RN-') THEN 'Routine maintenance'
        WHEN task_code = 'WATER TRX' THEN 'Water treatment'
        ELSE 'Other'
    END AS task_type,
    CURRENT_TIMESTAMP() AS _processed_at
FROM ranked
WHERE rn = 1
""")

StatementMeta(, , -1, Waiting, , Waiting, True)

## Service costs

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.service_cost
USING DELTA
AS
SELECT
    {company} AS company_key,
    s.source_record_area,
    s.source_row_id,
    s.vendor_code,
    s.vendor_name,
    s.work_number,
    s.invoice_number,
    s.journal_number,
    s.customer_code,
    s.location_code,
    s.technician_code,
    s.employee_code_text,
    s.employee_name,
    s.equipment_code,
    s.contract_number,
    s.contract_sequence,
    s.cost_code,
    s.cost_label,
    s.cost_description,
    s.document_code,
    s.document_date,
    s.document_amount,
    s.billing_amount,
    s.extended_cost,
    s.ws_extended_cost,
    s.committed_cost,
    s.transaction_quantity,
    s.actual_hours,
    s.billed_hours,
    s.transaction_source,
    s.transaction_description,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(SILVER_SCHEMA)}.service_cost_transaction s
WHERE COALESCE(s.document_amount, 0) <> 0
""")

StatementMeta(, , -1, Waiting, , Waiting, True)

## Service labor estimates

In [8]:
min_date = sql_string(MIN_SERVICE_DATE)
spark.sql(f"""
CREATE OR REPLACE TABLE {q(SILVER_SCHEMA)}.service_labor_estimate
USING DELTA
AS
WITH contracts AS (
    SELECT CUSTNMBR, ADRSCODE, Contract_Number, WSCONTSQ,
           Contract_Start_Date, Contract_Expiration_Date
    FROM {q(BRONZE_SCHEMA)}.SV00500
    UNION ALL
    SELECT CUSTNMBR, ADRSCODE, Contract_Number, WSCONTSQ,
           Contract_Start_Date, Contract_Expiration_Date
    FROM {q(BRONZE_SCHEMA)}.SV00501
),
service_task AS (
    SELECT
        {company} AS company_key,
        TRIM(sh.CUSTNMBR) AS customer_code,
        TRIM(sh.ADRSCODE) AS location_code,
        TRIM(sh.Contract_Number) AS contract_number,
        TRIM(st.Service_Call_ID) AS work_number,
        CAST(st.Estimate_Hours AS DOUBLE) / 100.0 AS estimated_hours,
        CAST(sh.Date1 AS DATE) AS scheduled_date,
        'SERVICE_CALL_TASK' AS estimate_source,
        ROW_NUMBER() OVER (
            PARTITION BY TRIM(st.Service_Call_ID), TRIM(st.Task_Code)
            ORDER BY
                CASE WHEN c.Contract_Number = sh.Contract_Number THEN 0 ELSE 1 END,
                c.WSCONTSQ DESC,
                c.Contract_Start_Date DESC
        ) AS rn
    FROM {q(BRONZE_SCHEMA)}.SV00302 st
    INNER JOIN {q(BRONZE_SCHEMA)}.SV00300 sh
      ON st.Service_Call_ID = sh.Service_Call_ID
    LEFT JOIN contracts c
      ON c.CUSTNMBR = sh.CUSTNMBR
     AND c.ADRSCODE = sh.ADRSCODE
     AND c.Contract_Number = sh.Contract_Number
     AND sh.Date1 BETWEEN c.Contract_Start_Date AND c.Contract_Expiration_Date
    WHERE COALESCE(st.Estimate_Hours, 0) > 0
      AND COALESCE(TRIM(st.Task_Code), '') <> 'DEFAULT'
      AND COALESCE(st.Task_Description, '') NOT LIKE '%DEFAULT%'
      AND CAST(sh.Date1 AS DATE) >= CAST({min_date} AS DATE)
),
pending_contract AS (
    SELECT
        {company} AS company_key,
        TRIM(cts.CUSTNMBR) AS customer_code,
        TRIM(cts.ADRSCODE) AS location_code,
        TRIM(cts.Contract_Number) AS contract_number,
        TRIM(cts.Service_Call_ID) AS work_number,
        CAST(cts.Estimate_Hours AS DOUBLE) / 100.0 AS estimated_hours,
        CAST(cts.Schedule_Date AS DATE) AS scheduled_date,
        'CONTRACT_TASK_SCHEDULE' AS estimate_source
    FROM {q(BRONZE_SCHEMA)}.SV00585 cts
    INNER JOIN contracts c
      ON cts.CUSTNMBR = c.CUSTNMBR
     AND cts.ADRSCODE = c.ADRSCODE
     AND cts.Contract_Number = c.Contract_Number
     AND cts.WSCONTSQ = c.WSCONTSQ
    WHERE COALESCE(cts.Estimate_Hours, 0) > 0
      AND cts.`Required` = 1
      AND CAST(c.Contract_Start_Date AS DATE) >= CAST({min_date} AS DATE)
      AND TRIM(cts.Service_Call_ID) = ''
)
SELECT company_key, customer_code, location_code, contract_number,
       work_number, estimated_hours, scheduled_date, estimate_source,
       CURRENT_TIMESTAMP() AS _processed_at
FROM service_task WHERE rn = 1
UNION ALL
SELECT company_key, customer_code, location_code, contract_number,
       work_number, estimated_hours, scheduled_date, estimate_source,
       CURRENT_TIMESTAMP() AS _processed_at
FROM pending_contract
""")

StatementMeta(, 85b89331-b16a-48c8-a299-071d8de3741e, 10, Finished, Available, Finished, False)

DataFrame[]

# Review

In [9]:
names = [
    "service_call", "service_call_hours", "service_call_task_budget", "service_appointment",
    "service_labor_actual", "service_labor_estimate",
    "service_call_equipment", "service_call_equipment_task", "service_cost"
]
for name in names:
    print(f"{SILVER_SCHEMA}.{name}: {spark.table(f'{SILVER_SCHEMA}.{name}').count():,} rows")

StatementMeta(, 85b89331-b16a-48c8-a299-071d8de3741e, 11, Finished, Available, Finished, False)

silver.service_call: 23,395 rows
silver.service_call_hours: 23,392 rows
silver.service_call_task_budget: 9,054 rows
silver.service_appointment: 48,516 rows
silver.service_labor_actual: 46,441 rows
silver.service_labor_estimate: 10,615 rows
silver.service_call_equipment: 115,592 rows
silver.service_call_equipment_task: 109,747 rows
silver.service_cost: 357 rows
